# Microphone Language Prediction

Record audio from your microphone, save it as a temporary `.wav` file, and classify it with the trained spoken-language identification model.

Run the cells from top to bottom. The recording cell records for a fixed number of seconds, then the prediction cell reuses `src/predict.py`.

In [212]:
# Notebook-only dependency for microphone recording.
# This intentionally does not modify requirements.txt.
%pip install sounddevice

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [213]:
from datetime import datetime
from pathlib import Path
import sys

import sounddevice as sd
import soundfile as sf
import torch
from IPython.display import Audio, display


def find_project_root(start: Path) -> Path:
    """Find the project root from either the repo root or notebooks folder."""
    for path in [start, *start.parents]:
        if (path / "src" / "predict.py").exists():
            return path
    raise RuntimeError("Could not find project root containing src/predict.py")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SRC_DIR = PROJECT_ROOT / "src"
sys.path.insert(0, str(SRC_DIR))

from config import CHECKPOINT_DIR, LANGUAGES, MODEL_TYPE, SAMPLE_RATE
from predict import load_model, predict

# Change this to "lstm" to test the LSTM checkpoint.
MODEL_TO_TEST = MODEL_TYPE  # options: "cnn", "lstm"

print("Project root:", PROJECT_ROOT)
print("Languages:", ", ".join(lang.capitalize() for lang in LANGUAGES))
print("Model to test:", MODEL_TO_TEST)
print("Checkpoint:", CHECKPOINT_DIR / MODEL_TO_TEST / "best_model.pth")

Project root: C:\Users\user\Documents\Projects\language_detection
Languages: Amharic, Arabic, Chinese, English, French, Hindi, Italian, Russian, Spanish
Model to test: cnn
Checkpoint: C:\Users\user\Documents\Projects\language_detection\checkpoints\cnn\best_model.pth


In [214]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = load_model(device, MODEL_TO_TEST)

print(f"Model loaded: {MODEL_TO_TEST}")
print("Device:", device)

Model loaded: cnn
Device: cpu


In [215]:
# Optional: inspect available audio devices if recording does not use the microphone you expect.
sd.query_devices()

   0 Microsoft Sound Mapper - Input, MME (2 in, 0 out)
>  1 Microphone Array (Realtek(R) Au, MME (2 in, 0 out)
   2 Microsoft Sound Mapper - Output, MME (0 in, 2 out)
<  3 Speakers (Realtek(R) Audio), MME (0 in, 2 out)
   4 Primary Sound Capture Driver, Windows DirectSound (2 in, 0 out)
   5 Microphone Array (Realtek(R) Audio), Windows DirectSound (2 in, 0 out)
   6 Primary Sound Driver, Windows DirectSound (0 in, 2 out)
   7 Speakers (Realtek(R) Audio), Windows DirectSound (0 in, 2 out)
   8 Speakers (Realtek(R) Audio), Windows WASAPI (0 in, 2 out)
   9 Microphone Array (Realtek(R) Audio), Windows WASAPI (2 in, 0 out)
  10 Microphone (Realtek HD Audio Mic input), Windows WDM-KS (2 in, 0 out)
  11 Stereo Mix (Realtek HD Audio Stereo input), Windows WDM-KS (2 in, 0 out)
  12 Headphones (Realtek HD Audio 2nd output), Windows WDM-KS (0 in, 2 out)
  13 Speakers 1 (Realtek HD Audio output with HAP), Windows WDM-KS (0 in, 2 out)
  14 Speakers 2 (Realtek HD Audio output with HAP), Windows WDM

In [216]:
RECORDINGS_DIR = PROJECT_ROOT / "notebooks" / "recordings"
RECORDINGS_DIR.mkdir(parents=True, exist_ok=True)

def record_audio(seconds: float = 7.0, device_id=None) -> Path:
    """Record microphone audio and save it as a WAV file."""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_path = RECORDINGS_DIR / f"microphone_{timestamp}.wav"

    print(f"Recording for {seconds:.1f} seconds at {SAMPLE_RATE} Hz...")
    print("Speak now.")

    audio = sd.rec(
        int(seconds * SAMPLE_RATE),
        samplerate=SAMPLE_RATE,
        channels=1,
        dtype="float32",
        device=device_id,
    )
    sd.wait()

    sf.write(output_path, audio, SAMPLE_RATE)
    print("Saved recording to:", output_path)

    return output_path

In [223]:
# Change seconds if you want a longer/shorter recording.
# If needed, pass device_id from sd.query_devices(), for example: record_audio(seconds=5, device_id=1)
audio_path = record_audio(seconds=7)
display(Audio(filename=str(audio_path)))

Recording for 7.0 seconds at 16000 Hz...
Speak now.
Saved recording to: C:\Users\user\Documents\Projects\language_detection\notebooks\recordings\microphone_20260503_115043.wav


In [224]:
predicted_language, confidences = predict(str(audio_path), model, device)

print(f"Predicted language: {predicted_language.capitalize()}")
print(f"Confidence: {confidences[predicted_language]:.1%}")
print("\nAll scores:")

for lang, prob in sorted(confidences.items(), key=lambda item: -item[1]):
    bar = "#" * int(prob * 40)
    print(f"{lang.capitalize():12s} {prob:6.1%} {bar}")

Predicted language: Spanish
Confidence: 99.5%

All scores:
Spanish       99.5% #######################################
Italian        0.2% 
Russian        0.1% 
Arabic         0.1% 
French         0.1% 
English        0.0% 
Amharic        0.0% 
Hindi          0.0% 
Chinese        0.0% 


## Notes

- The notebook uses the language list in `src/config.py`.
- Set `MODEL_TO_TEST = "cnn"` or `MODEL_TO_TEST = "lstm"` before loading the model.
- CNN checkpoints load from `checkpoints/cnn/best_model.pth`; LSTM checkpoints load from `checkpoints/lstm/best_model.pth`.
- If `LANGUAGES` changed after the model was trained, retrain before using this notebook.
- If recording fails, run the audio device cell and pass the correct `device_id` into `record_audio()`.